## 1. Setup
Mount Google Drive, check the GPU and make sure the datasets are in the project folder.
### Step 1.1: Mount Google Drive and create the project folder.
Datasets, checkpoints and model files are stored on Drive so they survive a Colab disconnect.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/adtc-msme-project'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project folder ready at: {PROJECT_DIR}")
print("Contents:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
Project folder ready at: /content/drive/MyDrive/adtc-msme-project
Contents: ['training_data_v3_final.jsonl', 'training_data_v3_clean.jsonl', 'reformat_progress.json', 'training_data_v3_reformatted.jsonl', 'training_data_v4_merged.jsonl', 'msme-qwen2.5-1.5b-lora', 'msme-qwen2.5-1.5b-merged', 'msme-qwen2.5-1.5b-f16.gguf', 'msme-qwen2.5-1.5b-Q4_K_M.gguf', 'training_data_v5.jsonl', 'msme-qwen2.5-1.5b-merged-v6']


### Step 1.2: Confirm a GPU is attached (this run used a Tesla T4).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU name: Tesla T4


### Step 1.3: List the files in the project folder (quick check that the datasets are there).

In [ ]:
import os
print(os.listdir(PROJECT_DIR))

['training_data_v3_final.jsonl', 'training_data_v3_clean.jsonl', 'reformat_progress.json', 'training_data_v3_reformatted.jsonl', 'training_data_v4_merged.jsonl', 'msme-qwen2.5-1.5b-lora', 'msme-qwen2.5-1.5b-merged', 'msme-qwen2.5-1.5b-f16.gguf', 'msme-qwen2.5-1.5b-Q4_K_M.gguf', 'training_data_v5.jsonl', 'msme-qwen2.5-1.5b-merged-v6']


### Step 1.4: Move training_data_v3_final.jsonl from My Drive into the project folder if it is not already there.

In [ ]:
import os, shutil

MYDRIVE = '/content/drive/MyDrive'
SRC = f'{MYDRIVE}/training_data_v3_final.jsonl'
DST = f'{PROJECT_DIR}/training_data_v3_final.jsonl'

if os.path.exists(SRC):
    shutil.move(SRC, DST)
    print("Moved into project folder.")
elif os.path.exists(DST):
    print("Already in project folder.")
else:
    print("Not found at either location — check the exact filename in Drive.")

print(os.listdir(PROJECT_DIR))

Already in project folder.
['training_data_v3_final.jsonl', 'training_data_v3_clean.jsonl', 'reformat_progress.json', 'training_data_v3_reformatted.jsonl', 'training_data_v4_merged.jsonl', 'msme-qwen2.5-1.5b-lora', 'msme-qwen2.5-1.5b-merged', 'msme-qwen2.5-1.5b-f16.gguf', 'msme-qwen2.5-1.5b-Q4_K_M.gguf', 'training_data_v5.jsonl', 'msme-qwen2.5-1.5b-merged-v6']


## 2. Data check
Quality scan of the first run's dataset (v4).

### Step 2.1: Data quality check on training_data_v4_merged.jsonl (the first run's dataset, 3,308 records).
Scans for corrupted or refusal-style answers, then counts answers that still have no structure (bold, bullets, numbering or headers).

In [ ]:
import json

with open(f'{PROJECT_DIR}/training_data_v4_merged.jsonl') as f:
    final_records = [json.loads(line) for line in f]

print(f"Total records: {len(final_records)}")

# Broader corruption scan — same signals as before, run against the FINAL merged set
broader_signals = [
    "cannot generate", "cannot process this request", "not readable",
    "unreadable", "corrupted", "does not contain any discernible",
    "please provide a properly formatted", "i understand. to help you create",
    "provide a clear, legible", "provide a clear, readable",
    "i can't generate", "i can't provide", "i can't complete",
    "i cannot generate", "i cannot provide", "i cannot complete",
    "doesn't contain regulatory", "doesn't contain information about",
    "doesn't contain actionable", "doesn't contain substantive",
    "no bearing on your", "this text doesn't contain",
    "the source material i have on hand", "please provide the full",
    "please supply appropriate", "i appreciate the question, but the source",
    "i can't reformat", "i cannot reformat", "unable to reformat",
    "here's the reformatted", "here is the reformatted"  # catch any leftover preamble the reformatting step might have added
]

final_corrupted = []
for idx, record in enumerate(final_records):
    for m in record['messages']:
        content_lower = m['content'].lower()
        if any(phrase in content_lower for phrase in broader_signals):
            final_corrupted.append(idx)
            break

print(f"Corrupted/problematic records found: {len(final_corrupted)}")
if final_corrupted:
    print("Indices:", final_corrupted[:30])

# Structure check — confirm reformatting actually stuck
import re
bold_pattern = re.compile(r'\*\*[^*]+\*\*')
bullet_pattern = re.compile(r'(^|\n)\s*[-*•]\s', re.MULTILINE)
numbered_pattern = re.compile(r'(^|\n)\s*\d+\.\s', re.MULTILINE)
header_pattern = re.compile(r'(^|\n)#{1,4}\s', re.MULTILINE)

def has_structure(text):
    return bool(bold_pattern.search(text) or bullet_pattern.search(text)
                or numbered_pattern.search(text) or header_pattern.search(text))

plain_count = 0
for record in final_records:
    assistant_msgs = [m['content'] for m in record['messages'] if m['role'] == 'assistant']
    text = ' '.join(assistant_msgs)
    if not has_structure(text):
        plain_count += 1

print(f"Still plain prose (no structure): {plain_count} ({100*plain_count/len(final_records):.1f}%)")

Total records: 3308
Corrupted/problematic records found: 0
Still plain prose (no structure): 21 (0.6%)


## 3. Environment
Install the training libraries. Step 3.3 pins the versions used for training.

### Step 3.1 (superseded): Early install of peft and torchao.


In [ ]:
!pip install -U peft torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


### Step 3.2 (superseded): Install the latest transformers, peft, accelerate, trl, datasets, torchao and bitsandbytes.

In [ ]:
!pip install -q -U transformers peft accelerate trl datasets torchao bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.1 MB/s eta 0:00:00


### Step 3.3: Pin the training stack.

In [ ]:
!pip uninstall -y transformers peft accelerate bitsandbytes trl
!pip install -q "transformers==4.55.4" "accelerate==1.15.0" "bitsandbytes==0.50.2" "trl==0.13.0" datasets torchao peft

Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: peft 0.21.0
Uninstalling peft-0.21.0:
  Successfully uninstalled peft-0.21.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2
Found existing installation: trl 1.13.0
Uninstalling trl-1.13.0:
  Successfully uninstalled trl-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.9 MB/s eta 0:00:00
ERROR: pip


### Step 3.4: Upgrade peft and torchao again after the pinned install.

In [ ]:
!pip install -U peft torchao


## 4. LoRA fine-tuning (QLoRA)
Load the base model in 4-bit, attach the LoRA adapter, prepare the data and train.
### Step 4.1: Load Qwen2.5-1.5B-Instruct in 4-bit (NF4) and attach a LoRA adapter (QLoRA).




In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print("Confirmed adapter dtype:", next(p.dtype for p in model.parameters() if p.requires_grad))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Confirmed adapter dtype: torch.float32


### Step 4.2: Prepare the data and the trainer.


In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

DATA_PATH = f'{PROJECT_DIR}/training_data_v5.jsonl'

dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Convert conversational `messages` format into a flat `text` column
# using the tokenizer's own chat template (same one baked into the model)
def formatting_func(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

train_dataset = train_dataset.map(formatting_func)
eval_dataset = eval_dataset.map(formatting_func)

# Sanity check — confirm the conversion actually produced real text
print("Sample formatted example:")
print(train_dataset[0]["text"][:500])

# Output directory now lives on Drive — survives any Colab disconnect
OUTPUT_DIR = f'{PROJECT_DIR}/msme-qwen2.5-1.5b-lora'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,  # reduced from 3 — eval_loss plateaued after epoch 2 last time
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,  # avoid AMP/GradScaler dtype crash from before
    bf16=False,
    max_seq_length=1024,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer initialized — output will save directly to Drive")

Generating train split: 0 examples [00:00, ? examples/s]

Train: 3429 | Eval: 181


Map:   0%|          | 0/3429 [00:00<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

Sample formatted example:
<|im_start|>system
You are a helpful assistant advising Kenyan MSME operators on tax, registration, financing, and regulatory compliance.<|im_end|>
<|im_start|>user
I run a matatu transport business in Meru County with 8 employees and a fleet of 3 vehicles. I'm looking to expand but I'm worried about getting a loan right now. What should I know about MSME lending from banks versus other lenders in Kenya at the moment?<|im_end|>
<|im_start|>assistant
The MSME lending landscape in Kenya right now 


Map:   0%|          | 0/3429 [00:00<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

Trainer initialized — output will save directly to Drive


### Step 4.3: Train.

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,1.523500,1.506432
100,1.465500,1.423845
150,1.385000,1.384555
200,1.389600,1.351186


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss,Validation Loss
50,1.523500,1.506432
100,1.465500,1.423845
150,1.385000,1.384555
200,1.389600,1.351186
250,1.195700,1.344752
300,1.216300,1.328939
350,1.215400,1.322561
400,1.172300,1.319357


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=430, training_loss=1.3378484437632006, metrics={'train_runtime': 3566.0965, 'train_samples_per_second': 1.923, 'train_steps_per_second': 0.121, 'total_flos': 1.8127143036893184e+16, 'train_loss': 1.3378484437632006, 'epoch': 2.0})

### Step 4.4: Record the peft and torchao versions (for the provenance record).

In [ ]:
import peft, torchao
print("peft version:", peft.__version__)
print("torchao version:", torchao.__version__)

peft version: 0.21.0
torchao version: 0.18.0


## 5. Merge the adapter into the base model
Merging needs the base model in full precision, not 4-bit.

### Step 5.1: Merge the LoRA adapter (checkpoint-430) into the full-precision base model.


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = f"{OUTPUT_DIR}/checkpoint-430"  # your final checkpoint from tonight's run
MERGED_OUTPUT = f"{PROJECT_DIR}/msme-qwen2.5-1.5b-merged-v6"

# Load base model fresh (not quantized this time — merging needs full precision)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load your new adapter on top of the base model
model_with_adapter = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Merge adapter weights into the base model permanently
merged_model = model_with_adapter.merge_and_unload()

# Save the merged, full model
merged_model.save_pretrained(MERGED_OUTPUT)
tokenizer.save_pretrained(MERGED_OUTPUT)

print(f"Merged model saved to: {MERGED_OUTPUT}")

Merged model saved to: /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-merged-v6


## 6. Convert to GGUF
Get llama.cpp and convert the merged model to a 16-bit GGUF file.

### Step 6.1: Download llama.cpp.


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp.git

Cloning into 'llama.cpp'...
remote: Enumerating objects: 125516, done.
remote: Counting objects: 100% (795/795), done.
remote: Compressing objects: 100% (341/341), done.
remote: Total 125516 (delta 601), reused 454 (delta 454), pack-reused 124721 (from 3)
Receiving objects: 100% (125516/125516), 433.33 MiB | 11.04 MiB/s, done.
Resolving deltas: 100% (88618/88618), done.
Updating files: 100% (3602/3602), done.


### Step 6.2: Convert the merged model to 16-bit (f16) GGUF.

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py \
  /content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-merged-v6 \
  --outfile /content/msme-qwen2.5-1.5b-v6-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: msme-qwen2.5-1.5b-merged-v6
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  t

## 7. Write the digest into the chat template
The digest is a text of verified facts stored in the GGUF's chat template. This changes metadata only, not the weights.

### Step 7.1: Upload the digest file current_full_digest_template.jinja.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving current_full_digest_template.jinja to current_full_digest_template.jinja


### Step 7.2: Install the gguf Python package.

In [ ]:
!pip install -q gguf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.3 MB/s eta 0:00:00


### Step 7.3 (unused draft): Load the digest and copy the f16 file.

In [ ]:
from gguf import GGUFReader, GGUFWriter
import shutil

SOURCE_GGUF = "/content/msme-qwen2.5-1.5b-v6-f16.gguf"
DIGEST_FILE = "current_full_digest_template.jinja"
OUTPUT_GGUF = "/content/msme-qwen2.5-1.5b-v6-f16-patched.gguf"

# Load the new digest text
with open(DIGEST_FILE, "r") as f:
    new_template = f.read()

print(f"Loaded digest: {len(new_template)} chars")

# Read the source GGUF's existing metadata + tensors
reader = GGUFReader(SOURCE_GGUF)

# Copy the file first, then we'll patch just the chat_template field
shutil.copy(SOURCE_GGUF, OUTPUT_GGUF)
print(f"Copied to: {OUTPUT_GGUF}")

Loaded digest: 8620 chars
Copied to: /content/msme-qwen2.5-1.5b-v6-f16-patched.gguf


### Step 7.4: Locate llama.cpp's gguf_new_metadata.py script.

In [ ]:
!find llama.cpp -name "gguf_new_metadata.py"

llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py


### Step 7.5: Show the script's options

In [ ]:
!python llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py --help

usage: gguf_new_metadata.py [-h] [--general-name "name"]
                            [--general-description "Description ..."]
                            [--chat-template "{% ... %} ..."]
                            [--chat-template-config tokenizer_config.json]
                            [--chat-template-file chat_template.jinja]
                            [--pre-tokenizer "pre tokenizer"]
                            [--remove-metadata general.url]
                            [--special-token bos | eos | eot | eom | unknown | seperator | padding | mask | fim_pre | fim_suf | fim_mid | fim_pad | fim_rep | fim_sep | prefix | suffix | middle "<token>"]
                            [--special-token-by-id bos | eos | eot | eom | unknown | seperator | padding | mask | fim_pre | fim_suf | fim_mid | fim_pad | fim_rep | fim_sep | prefix | suffix | middle 0]
                            [--force] [--verbose]
                            input output

Make a copy of a GGUF file with new metadata


### Step 7.6: Write the digest into the GGUF's chat template.

In [ ]:
!python llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py \
  /content/msme-qwen2.5-1.5b-v6-f16.gguf \
  /content/msme-qwen2.5-1.5b-v6-f16-digest.gguf \
  --chat-template-file current_full_digest_template.jinja \
  --force

INFO:gguf-new-metadata:* Loading: /content/msme-qwen2.5-1.5b-v6-f16.gguf
INFO:gguf-new-metadata:* Writing: /content/msme-qwen2.5-1.5b-v6-f16-digest.gguf
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
Writing:   0% 0.00/3.09G [00:00<?, ?byte/s]INFO:gguf.gguf_writer:Writing the following files:
INFO:gguf.gguf_writer:/content/msme-qwen2.5-1.5b-v6-f16-digest.gguf: n_tensors = 338, total_size = 3.1G
Writing: 100% 3.09G/3.09G [00:27<00:00, 112Mbyte/s]


### Step 7.7: Verify the digest is in the file.

In [ ]:
from gguf import GGUFReader

r = GGUFReader('/content/msme-qwen2.5-1.5b-v6-f16-digest.gguf')
for f in r.fields.values():
    if f.name == 'tokenizer.chat_template':
        template = bytes(f.parts[-1]).decode('utf-8')
        print(f"Template length: {len(template)} chars")
        print("Contains 'Vuka loan':", "Vuka loan" in template)
        print("Contains anti-fabrication rule:", "Never state a Shilling total" in template)
        break

Template length: 8620 chars
Contains 'Vuka loan': True
Contains anti-fabrication rule: True


## 8. Quantize to Q4_K_M
Build llama.cpp's quantization tool, then shrink the f16 file to 4-bit (Q4_K_M).

### Step 8.1 (diagnostic): Check whether llama-quantize is already built.

In [ ]:
import os
print(os.path.exists("llama.cpp/build/bin/llama-quantize"))

False


### Step 8.2 (diagnostic): Check available memory before building.

In [ ]:
!cat /proc/meminfo | head -3

MemTotal:       13286936 kB
MemFree:         9229416 kB
MemAvailable:   10203072 kB


### Step 8.3: Build llama-quantize from the llama.cpp source.


In [ ]:
!rm -rf llama.cpp/build
!cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
!cmake --build llama.cpp/build --config Release -j 2 --target llama-quantize

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.1-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

### Step 8.4: Quantize the f16 file with the digest to Q4_K_M.

In [ ]:
!llama.cpp/build/bin/llama-quantize \
  /content/msme-qwen2.5-1.5b-v6-f16-digest.gguf \
  /content/msme-qwen2.5-1.5b-v6-Q4_K_M.gguf \
  Q4_K_M

version: 0.4.1-dev (build 11027, commit c77ae695c)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/msme-qwen2.5-1.5b-v6-f16-digest.gguf' to '/content/msme-qwen2.5-1.5b-v6-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 31 key-value pairs and 338 tensors from /content/msme-qwen2.5-1.5b-v6-f16-digest.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:   

## 9. Save and download
Back up the quantized model to Drive, then download it.
### Step 9.1: Copy the quantized model from local disk to Drive as a backup.

In [ ]:
import shutil
shutil.copy(
    "/content/msme-qwen2.5-1.5b-v6-Q4_K_M.gguf",
    "/content/drive/MyDrive/adtc-msme-project/msme-qwen2.5-1.5b-v6-Q4_K_M.gguf"
)
print("Backed up to Drive")

Backed up to Drive


### Step 9.2: Download the quantized model to your laptop.

In [ ]:
from google.colab import files
files.download("/content/msme-qwen2.5-1.5b-v6-Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>